# CodeGen LoRA Training on Kaggle

Train **text2sql**, **sql2nosql**, and **nosql2doc** LoRA adapters on Kaggle GPU, then push adapters to the Hugging Face Hub.

**Validation** (baseline eval, BERTScore, Ollama judge) is intended to run **locally** after you download the adapters.

## Before you run

1. **Settings → Accelerator → GPU** (T4 x2 recommended).
2. **Settings → Internet → On**.
3. **Add-ons → Secrets** → create secret `HF_TOKEN` with a Hugging Face write token ([settings/tokens](https://huggingface.co/settings/tokens)).
4. Edit the **Configuration** cell below (`RUN_VERSION`, `HF_ADAPTER_REPO`, `EPOCHS`, etc.).

## Local workflow after training

```bash
export HF_TOKEN=hf_...   # read token is enough for public repos
python scripts/download_adapters_from_hf.py \
  --repo-id your-username/codegen-lora-v3 \
  --run v3

python scripts/run_baseline_eval.py --version v3 --tend-config spider --mlflow
```

## Configuration

Only `HF_TOKEN` is read from Kaggle Secrets. All other values mirror `.env.example` and can be changed here.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# --- Training run knobs ---
RUN_VERSION = "v4"                    # checkpoint folder: models/checkpoints/<RUN_VERSION>/
TASKS = ["text2sql", "sql2nosql", "nosql2doc"]
EPOCHS = 5
MAX_SAMPLES = None                    # e.g. 64 for a quick smoke run; None = full TEND train
DRY_RUN = False                       # True = print plan only, no training

# Hugging Face Hub destination for adapters (create empty repo or let upload script create it)
HF_ADAPTER_REPO = "your-username/codegen-lora-v3"  # <-- change to your repo id
HF_ADAPTER_PRIVATE = False            # True for a private Hub repo
SKIP_HF_UPLOAD = False                # True to train only (no Hub push)

# --- Project source ---
REPO_URL = "https://github.com/nayanjha16/CodeGen-Implementations-May_26.git"
REPO_BRANCH = "Group-43"
PROJECT_DIR = Path("/kaggle/working/CodeGen-Implementations-May_26")

# --- Environment (from .env.example; HF_TOKEN comes from Kaggle secret only) ---
ENV_VARS = {
  "MODEL_NAME": "Salesforce/codegen-350M-multi",
  "BERTSCORE_MODEL_NAME": "distilbert-base-uncased",
  "LLM_PROVIDER": "ollama",
  "OLLAMA_BASE_URL": "http://localhost:11434",
  "OLLAMA_JUDGE_MODEL": "qwen3:8b",
  "OLLAMA_TIMEOUT": "120",
  "HF_JUDGE_MODEL": "Qwen/Qwen2.5-0.5B-Instruct",
  "MODELS_BASE_DIR": "models/base",
  "MODELS_CHECKPOINTS_DIR": "models/checkpoints",
  "RESULTS_DIR": "results",
  "HF_DATASET_REPO": "care2achieve/tend",
  "TEND_DATASET_ID": "care2achieve/tend",
  "TEND_CACHE_DIR": "data/cache/tend",
}

print(f"Run version: {RUN_VERSION}")
print(f"Tasks: {TASKS}")
print(f"Epochs: {EPOCHS}")
print(f"HF adapter repo: {HF_ADAPTER_REPO}")

## Load `HF_TOKEN` from Kaggle Secrets

In [ ]:
def load_hf_token() -> str:
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as exc:
        raise RuntimeError(
            "HF_TOKEN not found. Add it under Add-ons → Secrets in Kaggle."
        ) from exc
    if not token:
        raise RuntimeError("HF_TOKEN secret is empty.")
    return token


HF_TOKEN = load_hf_token()
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
print("HF_TOKEN loaded from Kaggle secret.")

## Clone project and install dependencies

In [ ]:
if PROJECT_DIR.exists():
    !rm -rf "{PROJECT_DIR}"

!git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} "{PROJECT_DIR}"
%cd "{PROJECT_DIR}"

os.environ["PYTHONPATH"] = str(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

!pip install -q -r requirements.txt

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Write `.env` and verify configuration

In [ ]:
env_path = PROJECT_DIR / ".env"
lines = [f"{key}={value}" for key, value in ENV_VARS.items()]
env_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"Wrote {env_path} ({len(lines)} vars, HF_TOKEN excluded)")

from src.utils.config import get_model_name, load_config
from src.utils.device import resolve_device

config = load_config()
print("Model:", get_model_name(config))
print("Device:", resolve_device(config.get("model", {}).get("device", "auto")))
print("TEND dataset:", config.get("datasets", {}).get("tend", {}).get("dataset_id"))
print("Training epochs (config default):", config.get("training", {}).get("epochs"))

## Optional dry run

In [ ]:
dry_cmd = [
    sys.executable,
    "scripts/train_all_lora.py",
    "--version", RUN_VERSION,
    "--tasks", *TASKS,
    "--epochs", str(EPOCHS),
    "--no-mlflow",
    "--dry-run",
]
if MAX_SAMPLES is not None:
    dry_cmd.extend(["--max-samples", str(MAX_SAMPLES)])

subprocess.run(dry_cmd, check=True, cwd=PROJECT_DIR)

## Train LoRA adapters

Trains all configured tasks sequentially. Held-out TEND `test` split is used for training-time eval loss only — full benchmark validation runs locally.

In [ ]:
if DRY_RUN:
    print("DRY_RUN=True — skipping training.")
else:
    train_cmd = [
        sys.executable,
        "scripts/train_all_lora.py",
        "--version", RUN_VERSION,
        "--tasks", *TASKS,
        "--epochs", str(EPOCHS),
        "--no-mlflow",
    ]
    if MAX_SAMPLES is not None:
        train_cmd.extend(["--max-samples", str(MAX_SAMPLES)])

    print("Running:", " ".join(train_cmd))
    result = subprocess.run(train_cmd, cwd=PROJECT_DIR)
    if result.returncode != 0:
        raise RuntimeError(f"Training failed with exit code {result.returncode}")

## Verify adapter artifacts

In [ ]:
from src.training.adapter_verify import verify_all_adapters
from src.utils.paths import get_models_checkpoints_dir

run_dir = get_models_checkpoints_dir() / RUN_VERSION
verify = verify_all_adapters(run=RUN_VERSION)
for task, res in verify.items():
    status = "OK" if res.ok else "FAIL"
    print(f"{task}: {status} -> {res.adapter_path}")
    if not res.ok:
        print(f"  missing={res.missing_files} errors={res.errors}")

failed = [t for t, r in verify.items() if not r.ok]
if failed:
    raise RuntimeError(f"Adapter verification failed: {failed}")

print(f"\nCheckpoint run ready: {run_dir}")

## Upload adapters to Hugging Face Hub

In [ ]:
if SKIP_HF_UPLOAD or DRY_RUN:
    print("Skipping Hugging Face upload.")
else:
    import json
    from datetime import datetime, timezone

    from huggingface_hub import HfApi, create_repo

    api = HfApi(token=HF_TOKEN)
    create_repo(
        HF_ADAPTER_REPO,
        repo_type="model",
        private=HF_ADAPTER_PRIVATE,
        exist_ok=True,
    )

    for task in TASKS:
        adapter_dir = run_dir / task
        print(f"Uploading {adapter_dir} -> {HF_ADAPTER_REPO}/{task}/")
        api.upload_folder(
            folder_path=str(adapter_dir),
            path_in_repo=task,
            repo_id=HF_ADAPTER_REPO,
            repo_type="model",
            commit_message=f"Upload {task} LoRA adapter ({RUN_VERSION})",
        )

    for summary_path in sorted(run_dir.glob("training_summary_*.json")):
        api.upload_file(
            path_or_fileobj=str(summary_path),
            path_in_repo=summary_path.name,
            repo_id=HF_ADAPTER_REPO,
            repo_type="model",
            commit_message=f"Upload training summary ({RUN_VERSION})",
        )

    manifest = {
        "checkpoint_run": RUN_VERSION,
        "repo_id": HF_ADAPTER_REPO,
        "tasks": TASKS,
        "uploaded_at": datetime.now(timezone.utc).isoformat(),
    }
    manifest_path = run_dir / "hf_upload_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    api.upload_file(
        path_or_fileobj=str(manifest_path),
        path_in_repo="hf_upload_manifest.json",
        repo_id=HF_ADAPTER_REPO,
        repo_type="model",
        commit_message=f"Upload manifest ({RUN_VERSION})",
    )

    print(f"\nHub URL: https://huggingface.co/{HF_ADAPTER_REPO}")

## Package outputs for Kaggle download

Copies the checkpoint run into `/kaggle/working/` so it appears in the notebook **Output** tab.

In [ ]:
import shutil

export_root = Path("/kaggle/working") / f"codegen-lora-{RUN_VERSION}"
if export_root.exists():
    shutil.rmtree(export_root)
shutil.copytree(run_dir, export_root)
print(f"Exported adapters to {export_root}")
print("Download from the Output tab, or pull from Hugging Face locally (recommended).")

## Local download & validation (run on your machine)

```bash
# 1. Download adapters from Hugging Face
export HF_TOKEN=hf_...   # optional for public repos
python scripts/download_adapters_from_hf.py \
  --repo-id your-username/codegen-lora-v3 \
  --run v3

# 2. Verify adapter files
python scripts/verify_lora_adapters.py --version v3

# 3. Run full validation (requires Ollama or set LLM_PROVIDER=huggingface in .env)
python scripts/run_baseline_eval.py --version v3 --tend-config spider --mlflow
```

Set in local `.env` for inference:

```bash
MODEL_ADAPTER_RUN=v3
# MODEL_ADAPTER=text2sql   # optional: load one task adapter at a time
```